# 02 - 检查扰动矩阵

检查 PerturbNet 数据中的扰动标签、multi-hot 扰动矩阵、GO 功能向量和蛋白质序列嵌入。

In [ ]:
import sys
sys.path.insert(0, '..')
from pathlib import Path
import anndata as ad
import numpy as np

# 查找数据文件
data_dirs = [Path('../data/example'), Path('../data/processed')]
h5ad_files = []
for d in data_dirs:
    if d.exists():
        h5ad_files.extend(list(d.glob('**/*.h5ad')))

print(f'Available h5ad files: {[f.name for f in h5ad_files]}')

if h5ad_files:
    adata = ad.read_h5ad(h5ad_files[0])
    print(f'\nLoaded: {h5ad_files[0].name}')
    print(f'Shape: {adata.shape}')

In [ ]:
# 检查扰动标签
if 'adata' in locals():
    print('obs columns:', list(adata.obs.columns))
    print()
    
    for col in ['perturbation', 'condition', 'perturb', 'guide', 'target', 'drug']:
        if col in adata.obs.columns:
            counts = adata.obs[col].value_counts()
            print(f'Perturbation labels ({col}):')
            print(f'  Total categories: {len(counts)}')
            print(f'  Top 20:')
            for k, v in counts.head(20).items():
                print(f'    {k}: {v}')
            
            # 单扰动 vs 组合扰动
            single = sum(c for k, c in counts.items() if '+' not in str(k))
            combo = sum(c for k, c in counts.items() if '+' in str(k))
            print(f'\n  Single perturbation cells: {single}')
            print(f'  Combinatorial perturbation cells: {combo}')
            break

In [ ]:
# 检查 obsm 中的扰动矩阵
if 'adata' in locals():
    print('obsm keys:', list(adata.obsm.keys()))
    print()
    
    for key in adata.obsm.keys():
        val = adata.obsm[key]
        print(f'{key}: shape={val.shape}, dtype={val.dtype if hasattr(val, "dtype") else type(val)}')
        
        # 如果是 multi-hot 扰动矩阵
        if 'target' in key.lower() or 'perturb' in key.lower():
            n_nonzero = (val > 0).sum(axis=1)
            print(f'  Mean perturbations per cell: {n_nonzero.mean():.2f}')
            print(f'  Cells with 0 perturbations: {(n_nonzero == 0).sum()}')
            print(f'  Cells with 1 perturbation: {(n_nonzero == 1).sum()}')
            print(f'  Cells with 2+ perturbations: {(n_nonzero >= 2).sum()}')
        print()

In [ ]:
# 检查 uns 中的额外信息 (SMILES, GO vector 等)
if 'adata' in locals():
    print('uns keys:', list(adata.uns.keys()))
    print()
    for key in adata.uns.keys():
        val = adata.uns[key]
        if isinstance(val, (str, int, float)):
            print(f'{key}: {val}')
        elif isinstance(val, (list, np.ndarray)):
            print(f'{key}: type={type(val).__name__}, len={len(val)}')
        elif isinstance(val, dict):
            print(f'{key}: dict with {len(val)} keys')
        else:
            print(f'{key}: {type(val).__name__}')